# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a reproducible workflow for loading, exploring, and analyzing the [FAIR²](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library. The dataset comprises mass spectrometry-based protein quantification and annotation results from stimulated human mast cells extracellular vesicles.

### Dataset Source
The dataset is described by a Croissant schema hosted at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset loaded: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review available record sets and fields, referencing their `@id` values as per the FAIR² Croissant metadata.

In [ ]:
# List all record sets and their IDs
print("Available record sets and fields (by @id):\n")
record_set_ids = []
for rs in metadata.record_set:
    print(f"RecordSet name: {rs.name}")
    print(f"  @id: {rs['@id']}")
    record_set_ids.append(rs['@id'])
    print("  Fields:")
    for field in getattr(rs, 'field', []):
        print(f"    Field name: {field.name}")
        print(f"      @id: {field['@id']}")
        print(f"      DataType: {field.data_type if hasattr(field, 'data_type') else 'Unknown'}")
    print("")
print(f"All RecordSet @id's: {record_set_ids}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. You should use the record set and field `@id`s listed above. 

The main data files for this dataset are usually available in the primary record set (e.g., the record set containing protein quantification results).

In [ ]:
# Here, we select all available record set @id's to extract
# If you know which one(s) have tabular data, you can restrict to those
from collections.abc import Iterable
record_sets = record_set_ids
dataframes = {}

for record_set_id in record_sets:
    print(f"\n--- Loading record set: {record_set_id} ---")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) == 0:
        print("  No records found for this RecordSet.")
        continue
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Loaded DataFrame shape: {df.shape}")
    print(f"  Columns: {df.columns.tolist()}")
    display(df.head(2)) # Only show preview if data exists

## 4. Exploratory Data Analysis (EDA)
We'll demonstrate basic EDA steps: filtering, normalization, and grouping, all referencing columns by their field `@id`. 

First, choose a numeric field for analysis. For FAIR², suppose the primary data record set contains fields like protein abundance or molecular weight. Replace `<numeric_field_id>` with the field `@id` as found above.

In [ ]:
# Replace these variables with appropriate @id values from above
# E.g. protein_abundance_field_id = 'cr:abundance'

# For demonstration, we select the first record set and attempt to select numeric fields
from pandas.api.types import is_numeric_dtype
record_set_for_analysis = None
numeric_field_id = None

for recset_id, df in dataframes.items():
    numeric_columns = [col for col in df.columns if is_numeric_dtype(df[col])]
    if numeric_columns:
        record_set_for_analysis = recset_id
        numeric_field_id = numeric_columns[0]
        print(f"Using RecordSet {recset_id} with numeric field {numeric_field_id}")
        break

if record_set_for_analysis is None or numeric_field_id is None:
    print("No numeric data found in loaded data.")
else:
    df = dataframes[record_set_for_analysis]
    threshold = df[numeric_field_id].mean()  # Choose a threshold, e.g., mean
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records in {record_set_for_analysis} with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Try grouping by another field, e.g., a categorical field
    group_field_id = None
    for col in df.columns:
        if col != numeric_field_id and df[col].dtype == object:
            group_field_id = col
            break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
    else:
        print("No suitable categorical field found for grouping.")

## 5. Visualization
Visualize the distribution of the numeric field and relationship with a grouping field if appropriate.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_for_analysis is not None and numeric_field_id is not None:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id], bins=30)
    plt.title(f"Distribution of {numeric_field_id} in {record_set_for_analysis}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id is not None:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
        plt.title(f"{numeric_field_id} grouped by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Using the `mlcroissant` library and the Croissant schema, you can easily explore and analyze complex biological datasets such as the FAIR² mass spectrometry experiment. All data operations reference entities by their `@id` fields to ensure reproducibility and schema consistency. 

**Next steps** may include more advanced statistical modeling, protein function annotation, or exporting filtered protein lists for downstream use.